# ChainScore — Benchmark vs. Aave Market Rates

**Research question:** DeFi lending protocols charge a single utilization-based rate to all borrowers, regardless of their creditworthiness. ChainScore assigns individual credit scores (PDs) to each wallet. This notebook asks:

> *What if Aave priced credit risk the way a bank does? How large is the spread between what borrowers actually pay and what they should pay based on their PD?*

**Methodology:**
- Aave V2 historical supply rates sourced from DeFiLlama (USDC, DAI — 2022-2023)
- Borrow rates approximated as: `borrow_rate ≈ supply_rate / utilization`  
  (typical utilization: 70-85% for stablecoins in this period)
- Credit-adjusted "fair rate": `r_fair = r_market + credit_spread(PD, LGD)`
  where `credit_spread = PD × LGD / (1 - PD)` (standard bond pricing)
- LGD (Loss Given Default) for overcollateralized DeFi loans: 15% (conservative)

**Data sources:**
- Aave V2 rates: DeFiLlama Yields API (`yields.llama.fi/chart/{pool_id}`) — fetched live
- Credit scores: ChainScore LR model trained on Aave V2 liquidation events

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'font.family': 'monospace'})
print('Setup complete.')

In [ ]:
# ── Load models and scores ─────────────────────────────────────────────────
lr = joblib.load('../models/logistic_regression.pkl')
feature_cols = pd.read_json('../models/feature_columns.json', typ='series').tolist()

df = pd.read_parquet('../data/processed/feature_matrix.parquet')
X  = df[feature_cols]
pds = lr.predict_proba(X)[:, 1]

df = df[['wallet', 'label']].copy()
df['pd']    = pds
df['score'] = ((1 - pds) * 1000).round().astype(int)

print(f"Wallets scored: {len(df):,}")
print(f"Default rate:   {df['label'].mean():.1%}")
print(f"Mean PD:        {df['pd'].mean():.1%}")
print(f"Mean score:     {df['score'].mean():.0f}")

In [ ]:
# ── Load Aave V2 historical rates ──────────────────────────────────────────
def load_rates(path, asset):
    with open(path) as f:
        data = json.load(f)
    rates = pd.DataFrame(data)
    rates['date'] = pd.to_datetime(rates['timestamp']).dt.date
    rates['supply_apy'] = rates['apyBase'].clip(0)
    rates['asset'] = asset
    return rates[['date', 'asset', 'supply_apy', 'tvlUsd']].dropna(subset=['supply_apy'])

usdc = load_rates('../data/processed/aave_v2_usdc_rates.json', 'USDC')
dai  = load_rates('../data/processed/aave_v2_dai_rates.json',  'DAI')
rates = pd.concat([usdc, dai], ignore_index=True)

# Approximate borrow rate: supply / utilization
# Aave V2 stablecoin optimal utilization = 80%
UTILIZATION = 0.80
rates['borrow_apy'] = rates['supply_apy'] / UTILIZATION

print(f"Rate data points: {len(rates):,}")
print(rates.groupby('asset')[['supply_apy', 'borrow_apy']].agg(['mean', 'max']).round(2))

## Credit-Adjusted Fair Rate

Standard bond credit pricing formula:

$$r_{fair} = r_{market} + \frac{PD \times LGD}{1 - PD}$$

Where:
- $r_{market}$ = Aave pool borrow rate (same for all borrowers)
- $PD$ = probability of default from ChainScore
- $LGD$ = loss given default (15% for overcollateralized DeFi loans)

The **credit spread** = $r_{fair} - r_{market}$ is what a credit-aware lender would add.

In [ ]:
LGD = 0.15  # 15% — conservative for overcollateralized loans

# Use USDC average borrow rate as the reference market rate
market_rate_usdc = usdc['borrow_apy'].mean() / 100  # convert % to decimal
print(f"Avg Aave V2 USDC borrow rate (2022-2023): {market_rate_usdc:.2%}")

# Credit spread per wallet
df['credit_spread'] = (df['pd'] * LGD / (1 - df['pd'].clip(0, 0.999))) * 100  # in %
df['fair_rate']     = market_rate_usdc * 100 + df['credit_spread']
df['rate_gap']      = df['fair_rate'] - market_rate_usdc * 100  # how much they're underpaying

# Tier labels
bins   = [0, 300, 500, 650, 800, 1001]
labels = ['Very High', 'High', 'Medium', 'Low', 'Very Low']
df['tier'] = pd.cut(df['score'], bins=bins, labels=labels, right=False)

tier_stats = df.groupby('tier', observed=True).agg(
    n=('pd', 'count'),
    mean_pd=('pd', 'mean'),
    mean_score=('score', 'mean'),
    avg_spread=('credit_spread', 'mean'),
    avg_fair_rate=('fair_rate', 'mean'),
).round(2)

tier_stats['market_rate'] = round(market_rate_usdc * 100, 2)
tier_stats['underpaying_by'] = (tier_stats['avg_fair_rate'] - tier_stats['market_rate']).round(2)

print(f"\nMarket rate (uniform, Aave): {market_rate_usdc:.2%}")
print("\nCredit-adjusted fair rates by risk tier:")
print(tier_stats[['n', 'mean_score', 'mean_pd', 'market_rate', 'avg_fair_rate', 'underpaying_by']].to_string())

## Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── 1. Score distribution by default status ────────────────────────────────
ax = axes[0, 0]
bins_s = np.linspace(0, 1000, 41)
ax.hist(df[df['label']==0]['score'], bins=bins_s, alpha=0.6, color='#2563eb', label='Non-default', density=True)
ax.hist(df[df['label']==1]['score'], bins=bins_s, alpha=0.6, color='#dc2626', label='Default (liquidated)', density=True)
for x in [300, 500, 650, 800]:
    ax.axvline(x, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
ax.set_xlabel('ChainScore')
ax.set_ylabel('Density')
ax.set_title('Score Separation: Default vs Non-Default')
ax.legend(fontsize=9)

# ── 2. Fair rate vs market rate by tier ───────────────────────────────────
ax = axes[0, 1]
tiers = tier_stats.index.tolist()
x = np.arange(len(tiers))
width = 0.35
bars1 = ax.bar(x - width/2, tier_stats['market_rate'], width, label='Aave market rate (uniform)', color='#94a3b8', alpha=0.8)
bars2 = ax.bar(x + width/2, tier_stats['avg_fair_rate'], width, label='Credit-adjusted fair rate', color='#dc2626', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(tiers, fontsize=9)
ax.set_ylabel('Borrow Rate (%)')
ax.set_title('Market Rate vs. Credit-Adjusted Fair Rate by Tier')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# ── 3. Credit spread distribution ────────────────────────────────────────
ax = axes[1, 0]
ax.hist(df['credit_spread'].clip(0, 20), bins=50, color='#f59e0b', alpha=0.8, edgecolor='white', linewidth=0.3)
ax.axvline(df['credit_spread'].mean(), color='red', linestyle='--', linewidth=1.5, label=f"Mean: {df['credit_spread'].mean():.1f}%")
ax.set_xlabel('Credit Spread (%)')
ax.set_ylabel('Number of wallets')
ax.set_title('Distribution of Implied Credit Spreads')
ax.legend(fontsize=9)

# ── 4. Aave V2 rates over time + crisis markers ───────────────────────────
ax = axes[1, 1]
usdc_ts = usdc.copy()
usdc_ts['date'] = pd.to_datetime(usdc_ts['date'])
usdc_ts = usdc_ts.sort_values('date')
ax.plot(usdc_ts['date'], usdc_ts['borrow_apy'], color='#2563eb', linewidth=1.2, label='USDC borrow APY (approx.)')

crises = [('2022-05-09', 'LUNA', '#dc2626'), ('2022-11-08', 'FTX', '#f59e0b'), ('2023-03-11', 'USDC\ndepeg', '#8b5cf6')]
for date, label, color in crises:
    d = pd.Timestamp(date)
    ax.axvline(d, color=color, linestyle='--', linewidth=1, alpha=0.8)
    ax.text(d, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 8, label, color=color, fontsize=7, rotation=90, va='top', ha='right')

ax.set_ylabel('APY (%)')
ax.set_title('Aave V2 USDC Borrow Rate Over Time')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

fig.suptitle('ChainScore — Benchmark vs. Aave V2 Market Rates', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/benchmark_aave_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/benchmark_aave_rates.png')

## Key Findings

In [ ]:
high_risk = df[df['tier'].isin(['High', 'Very High'])]
low_risk  = df[df['tier'].isin(['Low', 'Very Low'])]

total_excess_spread = df['credit_spread'].sum()
high_risk_excess    = high_risk['credit_spread'].sum()

print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print(f"\nAave V2 USDC uniform borrow rate (avg 2022-2023): {market_rate_usdc:.2%}")
print(f"\nCredit-adjusted rates by tier:")
for tier in ['Very Low', 'Low', 'Medium', 'High', 'Very High']:
    if tier in tier_stats.index:
        row = tier_stats.loc[tier]
        print(f"  {tier:<12}: fair rate {row['avg_fair_rate']:.1f}%  (spread: +{row['underpaying_by']:.1f}pp above market)")

print(f"\nAdverse selection gap:")
print(f"  Very High risk wallets underpay by: {tier_stats.loc['Very High', 'underpaying_by']:.1f} percentage points")
print(f"  Very Low risk wallets overpay by:  +0.0 pp (they set the market rate, in effect)")
print(f"\n  {high_risk_excess/total_excess_spread:.1%} of total unpriced credit risk is concentrated")
print(f"  in {len(high_risk)/len(df):.1%} of wallets (High + Very High tiers)")
print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print("""
Aave's uniform pricing creates adverse selection: high-risk borrowers
receive an implicit subsidy from low-risk borrowers who face the same rate.

A credit-aware lending protocol using ChainScore could:
  1. Charge risk-adjusted rates → eliminate adverse selection
  2. Set lower collateral requirements for high-score borrowers
  3. Alert lenders when a counterparty's score deteriorates

This is the standard bank credit desk problem applied to DeFi:
FICO scores exist precisely because flat-rate lending is inefficient.
""")